# Convert .csv files to made in R to netcdfs

### imports

In [1]:
import sys
import os

import numpy as np
import pandas as pd
import xarray as xr
import scipy as sci
import h5py as h5

import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
from cmocean.tools import lighten

# regridding package
import xesmf as xe


# print versions of packages
print("python version =",sys.version[:5])
print("numpy version =", np.__version__)
print("pandas version =", pd.__version__)
print("xarray version =", xr.__version__)
print("scipy version =", sci.__version__)
print("h5py version =", h5.__version__)
print("matplotlib version =", sys.modules[plt.__package__].__version__)
print("cmocean version =", sys.modules[cmo.__package__].__version__)
print("cartopy version =", sys.modules[ccrs.__package__].__version__)
print("xesmf version =", xe.__version__)


wrkdir = "/g/data/es60/SEAPODYM/pearse_GAMs"
os.chdir(wrkdir)


python version = 3.10.
numpy version = 2.2.5
pandas version = 2.3.3
xarray version = 2025.4.0
scipy version = 1.15.3
h5py version = 3.15.1
matplotlib version = 3.10.6
cmocean version = v3.0.3
cartopy version = 0.24.1
xesmf version = 0.8.10


### load the data

In [3]:
%%time

df_HF = pd.read_csv("anomalous_values_GAM_predictions_HF.csv")
df_MF = pd.read_csv("anomalous_values_GAM_predictions_MF.csv")
df_LF = pd.read_csv("anomalous_values_GAM_predictions_LF.csv")
df_T = pd.read_csv("anomalous_values_GAM_predictions_T.csv")

df_HF

CPU times: user 14.2 s, sys: 1.74 s, total: 16 s
Wall time: 17.1 s


,lon,lat,season_year,season,sst,T_L1,T_L2,U_L3,zeu,O2_L2,adultTF_HF,adultTF_HF_se,adultTF_T_L1,adultTF_sst,adultTF_U_L3,adultTF_O2_L2,adultTF_zeu,adultTF_T_L2,adultTF_intercept
0,120.5,-19.5,1,DJF,-0.093900,-0.061707,NaN,NaN,0.122063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000761
1,120.5,-19.5,1,JJA,0.148375,-0.000282,NaN,NaN,-0.022017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000761
2,120.5,-19.5,1,MAM,0.073905,0.032154,NaN,NaN,0.031338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000761
3,120.5,-19.5,1,SON,-0.178293,-0.119760,NaN,NaN,-0.092449,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000761
4,120.5,-19.5,2,DJF,0.002868,0.077574,NaN,NaN,0.034434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1209595,239.5,19.5,62,SON,-0.064778,0.014985,0.003356,0.000390,1.081240,0.004795,-0.003550,0.000502,-0.002992,-0.000346,-0.002062,0.000540,0.000064,0.002007,-0.000761
1209596,239.5,19.5,63,DJF,0.252172,0.240080,0.103971,-0.000026,-1.262398,0.028055,0.001675,0.000472,0.001954,0.001510,-0.001504,-0.001901,0.002559,-0.000181,-0.000761
1209597,239.5,19.5,63,JJA,0.171326,-0.251603,-0.077289,0.000049,0.427817,-0.026060,-0.000729,0.000476,-0.007838,0.001055,-0.001608,0.003842,0.000835,0.003746,-0.000761
1209598,239.5,19.5,63,MAM,-0.341725,0.049017,0.086544,-0.000326,-2.237491,0.014742,-0.003294,0.000520,-0.002257,-0.002208,-0.001078,-0.000515,0.003335,0.000189,-0.000761


### convert the dataframe to netCDF files

In [6]:
%%time

def df_to_netcdf(df: pd.DataFrame, path: str,
                 lon="lon", lat="lat", year="season_year", season="season"):
    """
    Coordinates: lon, lat, year, season
    Variables: all other columns
    Writes: NetCDF to `path`
    """

    # keep only needed columns + variables
    coord_cols = [lon, lat, year, season]
    var_cols = [c for c in df.columns if c not in coord_cols]

    d = df[coord_cols + var_cols].copy()

    # ensure clean dtypes for coords
    d[lon] = d[lon].astype("float32")
    d[lat] = d[lat].astype("float32")
    d[year] = d[year].astype("int32")
    
    # season -> numeric code 1..4
    season_order = ["DJF", "MAM", "JJA", "SON"]
    d[season] = pd.Categorical(d[season], categories=season_order, ordered=True)
    d[season] = (d[season].cat.codes + 1).astype("int8")  # DJF=1, MAM=2, JJA=3, SON=4
    
    # numeric variables
    for c in var_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce").astype("float32")

    # ensure unique coord combos
    if d.duplicated(subset=coord_cols).any():
        d = d.groupby(coord_cols, as_index=False)[var_cols].mean()

    ds = d.set_index(coord_cols).to_xarray().sortby(coord_cols)

    # store mapping
    ds[season].attrs["mapping"] = "1=DJF, 2=MAM, 3=JJA, 4=SON"

    encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
    ds.to_netcdf(path, encoding=encoding)
    return ds


# write your four files
df_to_netcdf(df_HF, "/g/data/es60/SEAPODYM/pearse_GAMs/anomalous_values_GAM_predictions_HF.nc")
df_to_netcdf(df_MF, "/g/data/es60/SEAPODYM/pearse_GAMs/anomalous_values_GAM_predictions_MF.nc")
df_to_netcdf(df_LF, "/g/data/es60/SEAPODYM/pearse_GAMs/anomalous_values_GAM_predictions_LF.nc")
df_to_netcdf(df_T,  "/g/data/es60/SEAPODYM/pearse_GAMs/anomalous_values_GAM_predictions_T.nc")


CPU times: user 12 s, sys: 530 ms, total: 12.5 s
Wall time: 15 s


<xarray.Dataset> Size: 73MB
Dimensions:            (lon: 120, lat: 40, season_year: 63, season: 4)
Coordinates:
  * lon                (lon) float32 480B 120.5 121.5 122.5 ... 238.5 239.5
  * lat                (lat) float32 160B -19.5 -18.5 -17.5 ... 17.5 18.5 19.5
  * season_year        (season_year) int32 252B 1 2 3 4 5 6 ... 59 60 61 62 63
  * season             (season) int8 4B 1 2 3 4
Data variables: (12/15)
    sst                (lon, lat, season_year, season) float32 5MB 3.296 ... ...
    T_L1               (lon, lat, season_year, season) float32 5MB 2.104 ... ...
    T_L2               (lon, lat, season_year, season) float32 5MB nan ... -0...
    U_L3               (lon, lat, season_year, season) float32 5MB nan ... 0....
    zeu                (lon, lat, season_year, season) float32 5MB 0.01347 .....
    O2_L2              (lon, lat, season_year, season) float32 5MB nan ... -0...
    ...                 ...
    adultTF_sst        (lon, lat, season_year, season) float32 5MB nan ... 0....
    adultTF_U_L3       (lon, lat, season_year, season) float32 5MB nan ... -0...
    adultTF_O2_L2      (lon, lat, season_year, season) float32 5MB nan ... 0....
    adultTF_zeu        (lon, lat, season_year, season) float32 5MB nan ... 0....
    adultTF_T_L2       (lon, lat, season_year, season) float32 5MB nan ... 0....
    adultTF_intercept  (lon, lat, season_year, season) float32 5MB -0.0007614...